# Setup

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

!export PATH="/home/<name>/.local/bin:$PATH"

# Create a virtual environment
!/home/<name>/.local/bin/uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install prettyprint sympy numpy pandas matplotlib transformers accelerate vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [13]:
!/home/ugheewala/.local/bin/uv pip install --python .venv/bin/python \
    "numpy<2" \
    "torch==2.1.2+cu118" \
    "transformers==4.51.3" \
    "accelerate==0.34.2" \
    "huggingface_hub>=0.23.0" \
    "safetensors" \
    "sentencepiece" \
    "tqdm" \
    "pandas" \
    "matplotlib" \
    "sympy" \
    "antlr4-python3-runtime==4.11.1" \
    --extra-index-url https://download.pytorch.org/whl/cu118

Resolved 39 packages in 1.34s                                        
Prepared 2 packages in 3.36s                                             
Uninstalled 2 packages in 1.35s
░░░░░░░░░░░░░░░░░░░░ [0/2] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 5.12s                               
 - tokenizers==0.20.3
 + tokenizers==0.21.4
 - transformers==4.46.3
 + transformers==4.51.3


In [ ]:
!/home/ugheewala/.local/bin/uv pip install --python .venv/bin/python \
    "nvidia-cusparse-cu11" \
    "nvidia-cublas-cu11" \
    "nvidia-cuda-runtime-cu11" \
    "nvidia-cudnn-cu11"

In [1]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
import os
import sys
import json
import time
import csv
import subprocess
from pathlib import Path
from pprint import pprint

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
PUBLIC_DATA_PATH   = "data/public.jsonl"
PRIVATE_DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"

PROJECT_ROOT = Path.cwd()

RESULTS_DIR = PROJECT_ROOT / "results"
BASELINE1_DIR = RESULTS_DIR / "baseline1_weakest"

VAL_FRAC = 0.20
SPLIT_SEED = 414

CACHE_DIR = None
HF_HOME_DIR = None

MAX_INPUT_TOKENS = 16384
MAX_NEW_TOKENS_SMOKE = 512
MAX_NEW_TOKENS_BASELINE = 1024

BATCH_SIZE = 1
LOAD_IN_4BIT = True

BASELINE1_DIR.mkdir(parents=True, exist_ok=True)

MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

if HF_HOME_DIR is not None:
    os.environ["HF_HOME"] = str(HF_HOME_DIR)

if CACHE_DIR is not None:
    Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

print("HF_HOME      :", os.environ.get("HF_HOME"))
print("HF_HUB_CACHE :", os.environ.get("HF_HUB_CACHE"))
print("cache_dir    :", CACHE_DIR)

HF_HOME      : None
HF_HUB_CACHE : None
cache_dir    : None


In [3]:
import torch

print(f"CUDA_VISIBLE_DEVICES (Env): {os.environ.get('CUDA_VISIBLE_DEVICES')}")

cuda_available = torch.cuda.is_available()
print(f"Is CUDA available? {cuda_available}")

if cuda_available:
    print(f"Current Device: {torch.cuda.current_device()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("PyTorch still can't see the GPU.")
    device = torch.device("cpu")

CUDA_VISIBLE_DEVICES (Env): 0
Is CUDA available? True
Current Device: 0
Device Name: NVIDIA GeForce RTX 2080 Ti


In [4]:
import site

roots = [Path(p) for p in site.getsitepackages()]
matches = []

for root in roots:
    if root.exists():
        matches.extend(root.rglob("libcusparse.so*"))

for m in matches:
    print(m)

/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cusparse/lib/libcusparse.so.11
/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cu13/lib/libcusparse.so.12


In [5]:
wanted_libs = {
    "libcusparse.so",
    "libcublas.so",
    "libcudart.so",
    "libcudnn.so",
}

lib_dirs = []

for root in map(Path, site.getsitepackages()):
    if not root.exists():
        continue

    for lib in wanted_libs:
        for match in root.rglob(lib + "*"):
            lib_dir = str(match.parent)
            if lib_dir not in lib_dirs:
                lib_dirs.append(lib_dir)

VENV = Path("/home/ugheewala/private/CSE151B_Kaggle/.venv")
SITE = VENV / "lib/python3.11/site-packages"

cuda11_dirs = [
    SITE / "nvidia/cusparse/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cudnn/lib",
    SITE / "torch/lib",
]

cuda11_dirs = [str(p) for p in cuda11_dirs if p.exists()]

path_line = ":".join(cuda11_dirs)

print("Add this before starting the notebook/kernel:")
print(f'export LD_LIBRARY_PATH="{path_line}:$LD_LIBRARY_PATH"')

Add this before starting the notebook/kernel:
export LD_LIBRARY_PATH="/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cusparse/lib:/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cublas/lib:/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cuda_runtime/lib:/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cudnn/lib:/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/torch/lib:$LD_LIBRARY_PATH"


In [6]:
import os
import subprocess
from pathlib import Path

VENV = Path("/home/ugheewala/private/CSE151B_Kaggle/.venv")
SITE = VENV / "lib/python3.11/site-packages"

cuda11_dirs = [
    SITE / "nvidia/cusparse/lib",
    SITE / "nvidia/cublas/lib",
    SITE / "nvidia/cuda_runtime/lib",
    SITE / "nvidia/cudnn/lib",
    SITE / "torch/lib",
]

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = ":".join(str(p) for p in cuda11_dirs if p.exists()) + ":" + env.get("LD_LIBRARY_PATH", "")

subprocess.run(
    [str(VENV / "bin/python"), "-m", "bitsandbytes"],
    env=env,
)

The directory listed in your path is found to be non-existent: //0.0.0.0
The directory listed in your path is found to be non-existent: 8888/user/ugheewala
The directory listed in your path is found to be non-existent: /opt/conda/etc/xml/catalog file
The directory listed in your path is found to be non-existent: /etc/xml/catalog
The directory listed in your path is found to be non-existent: ghcr.io/ucsd-ets/scipy-ml-notebook
The directory listed in your path is found to be non-existent: /tree
The directory listed in your path is found to be non-existent: //10.96.0.1
The directory listed in your path is found to be non-existent: /user/ugheewala
The directory listed in your path is found to be non-existent: /user/ugheewala/oauth_callback
The directory listed in your path is found to be non-existent: ghcr.io/ucsd-ets/scipy-ml-notebook
The directory listed in your path is found to be non-existent: --xla_gpu_cuda_data_dir=/opt/conda/lib
The directory listed in your path is found to be non-e

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++ BUG REPORT INFORMATION ++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++++++++++ OTHER +++++++++++++++++++++++++++
CUDA specs: CUDASpecs(highest_compute_capability=(7, 5), cuda_version_string='118', cuda_version_tuple=(11, 8))
PyTorch settings found: CUDA_VERSION=118, Highest Compute Capability: (7, 5).
To manually override the PyTorch CUDA version please see: https://github.com/TimDettmers/bitsandbytes/blob/main/docs/source/nonpytorchcuda.mdx
Found duplicate CUDA runtime files (see below).

We select the PyTorch default CUDA runtime, which is 11.8,
but this might mismatch with the CUDA version that is needed for bitsandbytes.
To override this behavior set the `BNB_CUDA_VERSION=<version string, e.g. 122>` environmental variable.

For example, if you want to use the CUDA version 122,
    BNB_CUDA_VERSION=122 python ...

OR set the environmental variable in you

CompletedProcess(args=['/home/ugheewala/private/CSE151B_Kaggle/.venv/bin/python', '-m', 'bitsandbytes'], returncode=0)

In [6]:
import json
from pathlib import Path

kernel_json = Path("/home/ugheewala/.local/share/jupyter/kernels/cse151b/kernel.json")

with open(kernel_json, "r") as f:
    spec = json.load(f)

ld_library_path = (
    "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cusparse/lib:"
    "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cublas/lib:"
    "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cuda_runtime/lib:"
    "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cudnn/lib:"
    "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/torch/lib:"
    "${LD_LIBRARY_PATH}"
)

spec.setdefault("env", {})
spec["env"]["LD_LIBRARY_PATH"] = ld_library_path
spec["env"]["BNB_CUDA_VERSION"] = "118"

with open(kernel_json, "w") as f:
    json.dump(spec, f, indent=2)

print(kernel_json)
print(json.dumps(spec, indent=2))

/home/ugheewala/.local/share/jupyter/kernels/cse151b/kernel.json
{
  "argv": [
    "/home/ugheewala/private/CSE151B_Kaggle/.venv/bin/python",
    "-Xfrozen_modules=off",
    "-m",
    "ipykernel_launcher",
    "-f",
    "{connection_file}"
  ],
  "display_name": "Python (cse151b)",
  "language": "python",
  "metadata": {
    "debugger": true
  },
  "kernel_protocol_version": "5.5",
  "env": {
    "LD_LIBRARY_PATH": "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cusparse/lib:/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cublas/lib:/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cuda_runtime/lib:/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cudnn/lib:/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/torch/lib:${LD_LIBRARY_PATH}",
    "BNB_CUDA_VERSION": "118"
  }
}


In [4]:
import transformers

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

try:
    import bitsandbytes as bnb
    print("bitsandbytes:", bnb.__version__)
except Exception as e:
    print("bitsandbytes import failed:", repr(e))

Could not load bitsandbytes native library: libcusparse.so.11: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/bitsandbytes/cextension.py", line 85, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/bitsandbytes/cextension.py", line 72, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/ctypes/__init__.py", line 454, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/ctypes/__init__.py", line 376, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libcusparse.so.11: cannot open shared object file: No such file or directory

CUDA Setup faile

torch: 2.1.2+cu118
cuda: 11.8
cuda available: True
transformers: 4.51.3
bitsandbytes: 0.45.5


In [5]:
from transformers.utils import is_torch_available, is_bitsandbytes_available

print("is_torch_available:", is_torch_available())
print("is_bitsandbytes_available:", is_bitsandbytes_available())

is_torch_available: True
is_bitsandbytes_available: True


In [6]:
from transformers import AutoTokenizer
#from vllm import LLM, SamplingParams
from tqdm import tqdm

from baseline.datasets import load_public_splits, load_private_set
from baseline.generation import GenerationConfig
from baseline.prompt_sets import build_prompt_texts
from baseline.modeling import ModelConfig, load_transformers_model, predownload_model
from baseline.scoring import load_judger, score_one, summarize_results
from baseline.progress_viz import RunProgressDashboard
from prompting.prompt_chain import build_prompt_chain
from baseline.runner import run_problem_set

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices - present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [7]:
splits = load_public_splits(PUBLIC_DATA_PATH, val_frac=VAL_FRAC, seed=SPLIT_SEED)

train_set = splits["train"]
val_set = splits["val"]
public_set = splits["public"]
private_set = load_private_set(PRIVATE_DATA_PATH)

print("Train summary:")
pprint(train_set.summary())

print("\nValidation summary:")
pprint(val_set.summary())

print("\nPublic summary:")
pprint(public_set.summary())

print("\nPrivate summary:")
pprint(private_set.summary())

Train summary:
{'n': 901,
 'n_answered': 901,
 'n_free_form': 601,
 'n_mcq': 300,
 'name': 'public_train'}

Validation summary:
{'n': 225,
 'n_answered': 225,
 'n_free_form': 150,
 'n_mcq': 75,
 'name': 'public_val'}

Public summary:
{'n': 1126,
 'n_answered': 1126,
 'n_free_form': 751,
 'n_mcq': 375,
 'name': 'public'}

Private summary:
{'n': 943, 'n_answered': 0, 'n_free_form': 643, 'n_mcq': 300, 'name': 'private'}


In [8]:
prompt_chain = build_prompt_chain(strategy_name="baseline")

for label, problem_set in [("train", train_set), ("val", val_set), ("private", private_set)]:
    problem = problem_set.problems()[0]
    spec = prompt_chain.build_spec(problem)

    print("=" * 80)
    print(label, "id=", problem.id, "template=", spec.name)
    print("metadata:", spec.metadata)
    print("generation_hints:", spec.generation_hints)
    print(spec.to_messages()[0]["content"][:300])
    print("--- user ---")
    print(spec.to_messages()[-1]["content"][:500])

train id= 499 template= baseline_mcq
metadata: {'strategy_name': 'baseline', 'route_name': 'mcq', 'tags': []}
generation_hints: {'temperature': 0.6, 'top_p': 0.95}
You are an expert mathematician. Read the problem and the answer choices below, then select the single best answer. Output only the letter of your chosen option inside \boxed{}, e.g. \boxed{C}.
--- user ---
We now define an algorithm: The definition of a(n) is the least odd number k such that k * 2^n + 1 is a prime number. Given the input x_list (a series of values): [70, 71, 72, 73, 74, 75, 76, 77, 78, 79], determine the corresponding output sequence y_list.

Answer choices:
A. [44, 43, 129, 26, 63, 1, 90, 33, 22, 243]
B. [37, 35, 122, 19, 64, 10, 96, 26, 20, 245]
C. [38, 40, 128, 22, 71, 3, 91, 28, 14, 248]
D. [43, 37, 125, 21, 70, 9, 98, 27, 13, 246]
E. [39, 39, 127, 23, 67, 5, 93, 29, 15, 249]

val id= 990 template= baseline_free_form
metadata: {'strategy_name': 'baseline', 'route_name': 'free_form', 'tags': []}
generati

## 4. Modeling

In [9]:
model_config = ModelConfig(
    model_id=MODEL_ID,
    cache_dir=CACHE_DIR,
    gpu_id=GPU_ID,
    #load_in_4bit=LOAD_IN_4BIT,
    load_in_4bit=False,
    torch_dtype="float16",
    max_input_tokens=MAX_INPUT_TOKENS,
    reuse_loaded=True,
)

if "model_bundle" in globals() and model_bundle.config.cache_key() == model_config.cache_key():
    print("Reusing notebook-level model_bundle.")
else:
    t0 = time.perf_counter()
    model_bundle = load_transformers_model(model_config)
    print(f"Model load/reuse time: {time.perf_counter() - t0:.2f} sec")

print("Model device:", model_bundle.device())

RuntimeError: Failed to import transformers.models.qwen3.modeling_qwen3 because of the following error (look up to see its traceback):
module 'torch.library' has no attribute 'register_fake'

In [7]:
generation_config = GenerationConfig(
    max_new_tokens=1024,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
)

result = run_problem_set(
    problem_set=val_set.head(5),
    model_bundle=model_bundle,
    generation_config=generation_config,
    batch_size=1,
    score=True,
    output_jsonl_path="results/notebook_val5.jsonl",
)

result.summary

Generating:   0%|          | 0/5 [00:00<?, ?it/s]/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/bitsandbytes/backends/cpu/ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Generating:   0%|          | 0/5 [10:26<?, ?it/s]

KeyboardInterrupt


KeyboardInterrupt



In [ ]:
df = pd.DataFrame(result.scored_rows)
df[["id", "is_mcq", "correct", "response"]].head()

In [ ]:
pd.DataFrame([result.timings | result.generations])